# C02w1 — Frozen FZ118 comparison of the two completed batches
Run in a fresh kernel from `experiments/`. Reuses `final_texts_54.csv` and the original 27 final texts; no generation or detector fitting. Reports TPR, ROC AUC, and exact tree-path contributions relative to fixed same-genre humans. Raw comparisons retain each batch's genre mix; standardized comparisons use equal weights over shared genres. These are descriptive discovery diagnostics, not an equivalence test or the later adaptation-held-out evaluation.


In [ ]:
# Export every figure to reproduction/figures as well as the run directory.
from pathlib import Path as _FigurePath
def _export_figure(figure, filename, *args, **kwargs):
    original = _FigurePath(filename)
    original.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(original, *args, **kwargs)
    figure_dir = ROOT / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    try:
        suffix = original.relative_to(OUT).with_suffix("").as_posix().replace("/", "_")
    except ValueError:
        suffix = original.stem
    target = figure_dir / ("C02w1_" + suffix + original.suffix)
    figure.savefig(target, *args, **kwargs)

from pathlib import Path
import json, hashlib, pickle, re
from collections import Counter
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy, textstat
from sklearn.metrics import roc_curve, auc
from IPython.display import display
ROOT=Path.cwd() if (Path.cwd()/'results').exists() else Path.cwd().parent
BASE=ROOT/'results/causal/C02d2_fz118_ghostbuster_raid'
PRIOR=ROOT/'results/causal/C02e2_raid_feature_guided'
PREV=ROOT/'results/causal/C02s_relative_likelihood_sensitivity'
INPUT=ROOT/'results/causal/C02w_adaptive_camouflage_v1/final_texts_54.csv'
LATEST=PREV/'wps_repair_v1/downstream_v1/hyphen_repair_v1/texts.csv'
old=pd.read_csv(LATEST,keep_default_na=False)
data=pd.read_csv(INPUT,keep_default_na=False)
assert old.case_id.is_unique and data.case_id.is_unique
assert len(old)==27 and len(data)==54 and set(old.case_id)<=set(data.case_id)
assert data.pipeline_status.eq('complete').all() and data.text.str.strip().ne('').all()
assert data.source_id.is_unique
for r in old.itertuples(): assert data.set_index('case_id').loc[r.case_id,'text']==r.text, r.case_id
data['batch']=np.where(data.case_id.isin(old.case_id),'Original 27','New 27')
data=data.sort_values(['batch','case_id']).reset_index(drop=True)
assert data.groupby('batch').size().eq(27).all()
features=pd.read_parquet(BASE/'fz118_paired_features.parquet')
assert features.source_id.is_unique
human=pd.read_csv(PREV/'roc_discovery/human_scores_and_weights.csv')
assert human.source_id.is_unique
train_ids=set(features.loc[features.dataset.eq('raid') & features.split.eq('train'),'source_id'])
assert set(human.source_id).isdisjoint(train_ids)
assert set(human.source_id).isdisjoint(data.source_id)
# Include any genre absent from the old human cohort using held-out RAID test humans.
# Existing reference rows are preserved; additions are explicitly recorded.
missing=set(data.domain)-set(human.domain)
if missing:
    extra=features.loc[features.dataset.eq('raid') & features.split.eq('test') & features.label.eq(0) & features.domain.isin(missing),['source_id','domain']].copy()
    assert missing<=set(extra.domain), f'No held-out human reference for {missing-set(extra.domain)}'
    human=pd.concat([human,extra],ignore_index=True)
    print('Added held-out test human references for:',sorted(missing))
assert human.source_id.is_unique
nlp=spacy.load('en_core_web_sm')
normalize=lambda t:str(t).replace('\n',' ').replace('\r',' ')
SEED=20260909; B=2000
# Content-addressed output avoids overwriting results from different input files.
paths=[INPUT,LATEST,BASE/'fz118_paired_features.parquet',BASE/'fz118_feature_contract.json',BASE/'google-10000-english.txt',PRIOR/'fz118_raid_model_features.json',PRIOR/'fz118_raid_no_parentheses_detector.pkl',PREV/'roc_discovery/human_scores_and_weights.csv']
contract=dict(inputs={str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths},seed=SEED,bootstrap=B,spacy=nlp.meta)
nb=ROOT/'experiments/C02w1_compare_batches.ipynb'
contract['code']=hashlib.sha256(json.dumps([c['source'] for c in json.loads(nb.read_text())['cells'] if c['cell_type']=='code']).encode()).hexdigest()
run=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()[:12]
OUT=ROOT/'results/causal/C02w1_compare_batches'/run
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'contract.json').write_text(json.dumps(contract,indent=2))
display(pd.crosstab(data.domain,data.batch)); print('Output:',OUT)


In [ ]:
import re, pickle
from collections import Counter
import textstat
PUNCTUATION = [('comma', ','), ('period', '.'), ('colon', ':'), ('semicolon', ';'),
    ('question', '?'), ('exclamation', '!'), ('hyphen', '-'), ('quote', '"'),
    ('lparen', '('), ('rparen', ')'), ('lbracket', '['), ('rbracket', ']'), ('newline', '\n')]
google_words=[w.strip().lower() for w in (BASE/'google-10000-english.txt').read_text().splitlines() if w.strip()]
GOOGLE_TOP={n:set(google_words[:n]) for n in [100,1000,10000]}

def safe_div(a, b):
    return float(a) / float(b) if b else 0.0


def clean_word_tokens(span):
    return [
        token.text.lower()
        for token in span
        if token.is_alpha
    ]


def clean_pos_tokens(span):
    return [
        token.pos_
        for token in span
        if token.is_alpha
    ]


def ngram_counter(sequence, n):
    if len(sequence) < n:
        return Counter()
    return Counter(
        tuple(sequence[i:i+n])
        for i in range(len(sequence) - n + 1)
    )


def multiset_overlap(left, right, n):
    a = ngram_counter(left, n)
    b = ngram_counter(right, n)
    denominator = min(sum(a.values()), sum(b.values()))
    if denominator == 0:
        return 0.0
    shared = sum((a & b).values())
    return float(shared / denominator)


def overlap_histogram(sentences, token_fn, n):
    if len(sentences) < 2:
        return np.zeros(10, dtype=np.float64)

    values = [
        multiset_overlap(token_fn(left), token_fn(right), n)
        for left, right in zip(sentences[:-1], sentences[1:])
    ]

    # Include 1.0 in the final bin.
    bins = np.minimum((np.asarray(values) * 10).astype(int), 9)
    counts = np.bincount(bins, minlength=10).astype(np.float64)
    return counts / max(1, len(values))


def conjunction_overlap(sentences, n):
    values = []
    for sentence in sentences:
        words = clean_word_tokens(sentence)
        for i, word in enumerate(words):
            if word == "and":
                values.append(
                    multiset_overlap(words[:i], words[i+1:], n)
                )
    return float(np.mean(values)) if values else 0.0


def punctuation_count(text, symbol_name, symbol):
    if symbol_name == "quote":
        return text.count('"') + text.count("“") + text.count("”")
    return text.count(symbol)


def finite_readability(fn, text):
    try:
        value = float(fn(text))
        return value if np.isfinite(value) else 0.0
    except Exception:
        return 0.0


def extract_fz118(text):
    text = str(text)
    doc = nlp(text)
    sentences = [sent for sent in doc.sents if clean_word_tokens(sent)]
    words = [token.text.lower() for token in doc if token.is_alpha]

    n_characters = len(text)
    n_words = len(words)
    n_sentences = len(sentences)
    n_syllables = int(textstat.syllable_count(text))
    n_difficult = int(textstat.difficult_words(text))
    n_short = sum(len(word) <= 3 for word in words)
    n_long = sum(len(word) >= 7 for word in words)

    values = [
        n_characters,
        n_syllables,
        n_words,
        n_sentences,
        n_difficult,
        n_short,
        n_long,
        safe_div(n_characters, n_words),
        safe_div(n_syllables, n_words),
        safe_div(n_words, n_sentences),
        safe_div(n_difficult, n_words),
        safe_div(n_short, n_words),
        safe_div(n_long, n_words),
        finite_readability(textstat.automated_readability_index, text),
        finite_readability(textstat.coleman_liau_index, text),
        finite_readability(textstat.flesch_kincaid_grade, text),
        finite_readability(textstat.flesch_reading_ease, text),
        finite_readability(textstat.gunning_fog, text),
        finite_readability(textstat.lix, text),
        finite_readability(textstat.mcalpine_eflaw, text),
        finite_readability(textstat.rix, text),
        finite_readability(textstat.smog_index, text),
        safe_div(sum(token.is_stop for token in doc if token.is_alpha), n_words),
        safe_div(len(set(words)), n_words),
        safe_div(sum(word in GOOGLE_TOP[100] for word in words), n_words),
        safe_div(sum(word in GOOGLE_TOP[1000] for word in words), n_words),
        safe_div(sum(word in GOOGLE_TOP[10000] for word in words), n_words),
    ]

    punct_counts = [
        punctuation_count(text, name, symbol)
        for name, symbol in PUNCTUATION
    ]
    values += [safe_div(count, n_characters) for count in punct_counts]
    values += [safe_div(count, n_sentences) for count in punct_counts]

    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    paragraph_word_counts = [
        sum(token.is_alpha for token in nlp.make_doc(paragraph))
        for paragraph in paragraphs
    ]
    values += [
        len(paragraphs),
        float(np.mean(paragraph_word_counts)) if paragraph_word_counts else 0.0,
    ]

    for token_fn in [clean_word_tokens, clean_pos_tokens]:
        for n in [1, 2, 3]:
            values += overlap_histogram(sentences, token_fn, n).tolist()

    values += [conjunction_overlap(sentences, n) for n in [1, 2, 3]]

    vector = np.asarray(values, dtype=np.float64)
    assert vector.shape == (118,)
    assert np.isfinite(vector).all()
    return vector
# Exact tree-path contributions, summed across the frozen forest.
FEATURE_NAMES=json.loads((BASE/'fz118_feature_contract.json').read_text())['feature_names']
model_features=json.loads((PRIOR/'fz118_raid_model_features.json').read_text())
with (PRIOR/'fz118_raid_no_parentheses_detector.pkl').open('rb') as f:detector=pickle.load(f)
assert list(detector.classes_)==[0,1]
def contributions(X):
    a=np.asarray(X,dtype=np.float32); output=np.zeros_like(a,dtype=float);bias=0.
    for estimator in detector.estimators_:
        tree=estimator.tree_; value=tree.value[:,0,:].astype(float)
        prob=value[:,1]/value.sum(axis=1);bias+=prob[0]/len(detector.estimators_)
        for i,row in enumerate(a):
            node=0
            while tree.children_left[node]!=tree.children_right[node]:
                feature=tree.feature[node]
                child=tree.children_left[node] if row[feature]<=tree.threshold[node] else tree.children_right[node]
                output[i,feature]+=(prob[child]-prob[node])/len(detector.estimators_)
                node=child
    assert np.allclose(bias+output.sum(axis=1),detector.predict_proba(X)[:,1],atol=1e-8)
    return output


@lru_cache(maxsize=8192)
def vector_for(text): return extract_fz118(normalize(text))
def frame_for(texts):
    return pd.DataFrame([vector_for(t) for t in texts],columns=FEATURE_NAMES)


In [ ]:
# Reuse the established extractor and exact additive forest decomposition above.
print('Extracting 54 final texts...',flush=True)
X=frame_for(data.text.tolist())[model_features]
scores=detector.predict_proba(X)[:,1]
C=contributions(X)
HX=features.set_index('source_id').loc[human.source_id,model_features]
h=detector.predict_proba(HX)[:,1]
if 'AI_score' in human:
    valid=human.AI_score.notna()
    np.testing.assert_allclose(h[valid],human.loc[valid,'AI_score'],atol=1e-8)
HC=contributions(HX)
href=pd.DataFrame(HC,columns=model_features).groupby(human.domain.to_numpy()).mean()
gap=C-href.loc[data.domain].to_numpy()
data.assign(AI_score=scores).to_csv(OUT/'AI_scores.csv',index=False)
X.assign(case_id=data.case_id.to_numpy()).to_csv(OUT/'features.csv',index=False)
pd.DataFrame(C,columns=model_features).assign(case_id=data.case_id.to_numpy(),batch=data.batch.to_numpy()).to_csv(OUT/'case_contributions.csv',index=False)
pd.DataFrame(gap,columns=model_features).assign(case_id=data.case_id.to_numpy(),batch=data.batch.to_numpy()).to_csv(OUT/'case_contribution_gaps.csv',index=False)
human.assign(frozen_AI_score=h).to_csv(OUT/'human_reference.csv',index=False)
href.to_csv(OUT/'human_mean_contributions.csv')
print('Scoring complete.')


## Compare batches
Shared thresholds use a human reference weighted to the pooled 54-text genre mix. Standardized results use equal weights over genres present in both batches. Bootstrap intervals independently resample cases within each batch and genre, conditional on the fixed detector, human reference, thresholds, and observed genre counts. Feature intervals are pointwise; they are not simultaneous or causal attributions. Differences are New minus Original, not paired differences between unrelated documents.


In [ ]:
budgets=np.array([.01,.05,.10,.20,.30,.50,1.])
labels=['Original 27','New 27']
shared=sorted(set(data.loc[data.batch.eq(labels[0]),'domain']) & set(data.loc[data.batch.eq(labels[1]),'domain']))
assert shared
all_summary=[]; all_ops=[]; all_contrib=[]; all_differences=[]; all_rocs=[]
def savefig(fig,name):
    fig.tight_layout()
    for ext in ['png','pdf']:_export_figure(fig,OUT/f'{name}.{ext}',dpi=300,bbox_inches='tight')
    plt.show()
for mode in ['raw','shared_genres']:
    target=(data.domain.value_counts(normalize=True).to_dict() if mode=='raw' else {g:1/len(shared) for g in shared})
    hw=np.array([target.get(g,0)/int(human.domain.eq(g).sum()) for g in human.domain]); hw/=hw.sum()
    cuts=np.r_[np.inf,np.unique(h)[::-1],-np.inf]
    fprs=np.array([hw[h>t].sum() for t in cuts])
    indices=[np.flatnonzero(fprs<=b+1e-12)[-1] for b in budgets]
    thresholds=cuts[indices]
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    estimates={}; boots={}; means={}; gapboots={}
    for label in labels:
        ix=np.flatnonzero(data.batch.eq(label) & data.domain.isin(target))
        domains=data.iloc[ix].domain.to_numpy()
        masses=({g:np.mean(domains==g) for g in sorted(set(domains))} if mode=='raw' else target)
        aw=np.array([masses[g]/sum(domains==g) for g in domains])
        a=scores[ix]; z=gap[ix]
        f,t,th=roc_curve(np.r_[np.zeros(len(h)),np.ones(len(a))],np.r_[h,a],sample_weight=np.r_[hw,aw],drop_intermediate=False)
        # Weighted AUC per positive case, with half credit for ties.
        ap=np.array([hw[h<s].sum()+.5*hw[h==s].sum() for s in a])
        outcomes=np.column_stack([a,ap,*[(a>cut).astype(float) for cut in thresholds]])
        estimates[label]=aw@outcomes; means[label]=aw@z
        rng=np.random.default_rng(SEED+(label=='New 27'))
        boot=np.zeros((B,outcomes.shape[1])); gb=np.zeros((B,z.shape[1]))
        for g,mass in masses.items():
            gi=np.flatnonzero(domains==g)
            draws=rng.choice(gi,size=(B,len(gi)),replace=True)
            boot+=mass*outcomes[draws].mean(axis=1)
            gb+=mass*z[draws].mean(axis=1)
        boots[label]=boot; gapboots[label]=gb
        all_summary.append(dict(mode=mode,batch=label,cases=len(ix),mean_AI_score=estimates[label][0],AUC=auc(f,t)))
        for j,b in enumerate(budgets):
            lo,hi=np.quantile(boot[:,2+j],[.025,.975])
            all_ops.append(dict(mode=mode,batch=label,requested_FPR=b,achieved_FPR=fprs[indices[j]],threshold=thresholds[j],TPR=estimates[label][2+j],ci_low=lo,ci_high=hi))
        lo,hi=np.quantile(gb,[.025,.975],axis=0)
        for j,feature in enumerate(model_features):all_contrib.append(dict(mode=mode,batch=label,feature=feature,mean_gap=means[label][j],ci_low=lo[j],ci_high=hi[j]))
        for ax in axes:ax.step(f,t,where='post',label=f'{label} ({auc(f,t):.3f})')
        all_rocs.extend(dict(mode=mode,batch=label,FPR=x,TPR=y,threshold=q) for x,y,q in zip(f,t,th))
    for ax in axes:
        ax.plot([0,1],[0,1],'--',color='gray');ax.set(xlabel='Genre-weighted human FPR',ylabel='AI TPR');ax.legend()
    axes[1].set_xlim(0,.2);fig.suptitle(mode);savefig(fig,f'ROC_{mode}')
    delta=estimates[labels[1]]-estimates[labels[0]]
    lo,hi=np.quantile(boots[labels[1]]-boots[labels[0]],[.025,.975],axis=0)
    names=['mean_AI_score','AUC']+[f'TPR_at_{b:g}' for b in budgets]
    for j,name in enumerate(names):all_differences.append(dict(mode=mode,metric=name,new_minus_original=delta[j],ci_low=lo[j],ci_high=hi[j]))
    gd=means[labels[1]]-means[labels[0]]
    gl,gh=np.quantile(gapboots[labels[1]]-gapboots[labels[0]],[.025,.975],axis=0)
    pd.DataFrame(dict(feature=model_features,new_minus_original=gd,ci_low=gl,ci_high=gh)).to_csv(OUT/f'contribution_differences_{mode}.csv',index=False)
    rank=np.argsort(np.maximum(np.abs(means[labels[0]]),np.abs(means[labels[1]])))[-30:]
    fig,ax=plt.subplots(figsize=(10,10))
    for k,label in enumerate(labels):
        lo,hi=np.quantile(gapboots[label],[.025,.975],axis=0)
        y=np.arange(len(rank))+(k-.5)*.2
        ax.hlines(y,lo[rank],hi[rank],color=f'C{k}',alpha=.4)
        ax.scatter(means[label][rank],y,label=label,s=22)
    ax.set_yticks(np.arange(len(rank)),np.array(model_features)[rank]);ax.axvline(0,color='gray');ax.legend()
    ax.set_xlabel('Mean contribution minus fixed same-genre human reference')
    savefig(fig,f'contributions_{mode}')
for name,rows in [('summary',all_summary),('TPR_at_FPR',all_ops),('contributions',all_contrib),('batch_differences',all_differences),('ROC_points',all_rocs)]:
    table=pd.DataFrame(rows);table.to_csv(OUT/f'{name}.csv',index=False)
    if name in ['summary','TPR_at_FPR','batch_differences']:display(table)
print('Saved CSV tables and PNG/PDF figures to',OUT)
